### get `featureClass.P_glcm`
by H.Nishiyama with GitHub Copilot<br>
ref: https://groups.google.com/g/pyradiomics/c/s1tbbvPFgyk<br>
2026-06-25 22:18:15<br>
2026-06-26 12:25:02<br>
using modified glcm.py in workspace folder<br>
for checking inner process add parametet `weightingNorm` to this code<br>
2026-06-26 21:40:22<br>
I think `p_glcm` might be related to inner GLCM by each anglar-direction(2d:4), but not symmetrical.<br>
`sumP_glcm` and `raw_sumP_glcm` are sum of each GLCM before normarization.<br>
These two parameters were modified by GtHub Copilot to allow for external extraction from the local version of glcm.py in workspace folder.<br>
2026-06-28 04:39:52<br>
Modified the code to extract GLCM and angular information as `orginal_P_glcm` and `original_angles` immediately after the calculation.<br>  

#### memo
2026-06-28 06:13:35<br>
The internal arrangement of the matrix elements differs from the standard GLCM pattern; the layout appears to have been modified to improve computational speed.

To obtain a GLCM with the standard ordering, I applied `<matrix>.[:,:,:,::-1].transpose(0,3,2,1)`.

*The internal ordering of the GLCM was verified by comparing the results with angle-specific GLCM outputs obtained via Excel simulation and Julia (radiomics.jl).
Placed `normal_ordered_GLCM_sample.png` in workspace folder

In [1]:
import os
import numpy as np
from radiomics import featureextractor
#from radiomics.glcm import RadiomicsGLCM
from glcm import RadiomicsGLCM
# from radiomics import base
# import six
import SimpleITK as sitk


In [2]:
dataDir = './'

imageName = sitk.ReadImage(os.path.join(dataDir, 'forRadiomicsTest.nrrd'))
maskName = sitk.ReadImage(os.path.join(dataDir, 'Segmentation.seg.nrrd'))

In [3]:
# calculate parameter for GLCM
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings['distances'] = [1]
extractor.settings['symmetricalGLCM'] = True
extractor.settings['normalize'] = True
extractor.settings['label'] = 1  # target area's label
extractor.settings['binWidth'] = 1
extractor.settings['force2D'] = True # force 2D calculation


## Set weightingNorm to None
default parameter

In [4]:
# extractor.settings['weightingNorm'] = 'no_weighting'
extractor.settings['weightingNorm'] = None  # default

In [5]:

glcm_calc = RadiomicsGLCM(imageName, maskName, **extractor.settings)


In [6]:
# Check the GLCM coefficients
print("Ng (Num_grayLevels):", glcm_calc.coefficients['Ng'])
print("GrayLevels:", glcm_calc.coefficients['grayLevels'])


Ng (Num_grayLevels): 4
GrayLevels: [1 2 3 4]


In [7]:
glcm_calc.execute()  # calc GLCM

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


{'Autocorrelation': array(4.11543367),
 'ClusterProminence': array(12.55974719),
 'ClusterShade': array(1.99023407),
 'ClusterTendency': array(2.00913519),
 'Contrast': array(1.84375),
 'Correlation': array(0.04252529),
 'DifferenceAverage': array(1.03380102),
 'DifferenceEntropy': array(1.74432286),
 'DifferenceVariance': array(0.75839982),
 'Id': array(0.60246599),
 'Idm': array(0.56409439),
 'Idmn': array(0.90980792),
 'Idn': array(0.81845238),
 'Imc1': array(-0.04887176),
 'Imc2': array(0.39657997),
 'InverseVariance': array(0.47266511),
 'JointAverage': array(2.01817602),
 'JointEnergy': array(0.10470409),
 'JointEntropy': array(3.53612608),
 'MCC': array(0.26153283),
 'MaximumProbability': array(0.18112245),
 'SumAverage': array(4.03635204),
 'SumEntropy': array(2.40180216),
 'SumSquares': array(0.9632213)}

#### original inner glcm parameter

In [8]:
original_P_glcm = glcm_calc.orginal_P_glcm
original_angles = glcm_calc.original_angles
print("Original GLCM shape:", original_P_glcm.shape)
print("Original GLCM matrix:\n", original_P_glcm)
print("Original angles:\n", original_angles)

Original GLCM shape: (1, 4, 4, 4)
Original GLCM matrix:
 [[[[ 7.  7.  3.  8.]
   [ 5.  6.  8.  7.]
   [ 3.  6.  3.  3.]
   [ 1.  2.  2.  1.]]

  [[ 7. 12.  8.  7.]
   [10.  6.  6. 11.]
   [ 3.  2.  4.  1.]
   [ 2.  3.  3.  4.]]

  [[ 3.  3.  3.  2.]
   [ 1.  3.  3.  2.]
   [ 0.  0.  0.  2.]
   [ 2.  1.  1.  1.]]

  [[ 1.  0.  3.  3.]
   [ 1.  4.  1.  2.]
   [ 1.  0.  0.  1.]
   [ 2.  1.  1.  1.]]]]
Original angles:
 [[ 0  1  1]
 [ 0  1  0]
 [ 0  1 -1]
 [ 0  0  1]]


In [9]:
p_glcm = glcm_calc.P_glcm  # get GLCM matrix [i,j,angle] with normalization 
print("GLCM shape:", p_glcm.shape)
print("GLCM matrix:\n", p_glcm)

GLCM shape: (1, 4, 4, 4)
GLCM matrix:
 [[[[0.14285714 0.125      0.06122449 0.14285714]
   [0.12244898 0.16071429 0.16326531 0.125     ]
   [0.06122449 0.08035714 0.06122449 0.04464286]
   [0.02040816 0.01785714 0.05102041 0.03571429]]

  [[0.12244898 0.16071429 0.16326531 0.125     ]
   [0.20408163 0.10714286 0.12244898 0.19642857]
   [0.04081633 0.04464286 0.07142857 0.02678571]
   [0.03061224 0.0625     0.04081633 0.05357143]]

  [[0.06122449 0.08035714 0.06122449 0.04464286]
   [0.04081633 0.04464286 0.07142857 0.02678571]
   [0.         0.         0.         0.03571429]
   [0.03061224 0.00892857 0.01020408 0.01785714]]

  [[0.02040816 0.01785714 0.05102041 0.03571429]
   [0.03061224 0.0625     0.04081633 0.05357143]
   [0.03061224 0.00892857 0.01020408 0.01785714]
   [0.04081633 0.01785714 0.02040816 0.01785714]]]]


In [10]:
print(glcm_calc.sumP_glcm)
print(glcm_calc.raw_sumP_glcm)

[[ 98. 112.  98. 112.]]
[[ 98. 112.  98. 112.]]


In [11]:
val_glcm = p_glcm * glcm_calc.sumP_glcm
sum_val_glcm = np.sum(val_glcm,axis=(0, 1, 2))
print("sum_val_glcm:\n", sum_val_glcm)
print("val_glcm:\n", val_glcm)

sum_val_glcm:
 [ 98. 112.  98. 112.]
val_glcm:
 [[[[14. 14.  6. 16.]
   [12. 18. 16. 14.]
   [ 6.  9.  6.  5.]
   [ 2.  2.  5.  4.]]

  [[12. 18. 16. 14.]
   [20. 12. 12. 22.]
   [ 4.  5.  7.  3.]
   [ 3.  7.  4.  6.]]

  [[ 6.  9.  6.  5.]
   [ 4.  5.  7.  3.]
   [ 0.  0.  0.  4.]
   [ 3.  1.  1.  2.]]

  [[ 2.  2.  5.  4.]
   [ 3.  7.  4.  6.]
   [ 3.  1.  1.  2.]
   [ 4.  2.  2.  2.]]]]


#### Transpose and flip the internal GLCM parameters to convert them to the standard GLCM order.
angle 0(~180), 45(~225), 90(~270), 135(~315)
[ 0  0  1], [ 0  1  1], [ 0  1  0], [ 0  1 -1]

In [12]:
tp_p_glcm = p_glcm[:,:,:,::-1].transpose(0,3,2,1)
print("GLCM matrix:\n", tp_p_glcm)

GLCM matrix:
 [[[[0.14285714 0.125      0.04464286 0.03571429]
   [0.125      0.19642857 0.02678571 0.05357143]
   [0.04464286 0.02678571 0.03571429 0.01785714]
   [0.03571429 0.05357143 0.01785714 0.01785714]]

  [[0.06122449 0.16326531 0.06122449 0.05102041]
   [0.16326531 0.12244898 0.07142857 0.04081633]
   [0.06122449 0.07142857 0.         0.01020408]
   [0.05102041 0.04081633 0.01020408 0.02040816]]

  [[0.125      0.16071429 0.08035714 0.01785714]
   [0.16071429 0.10714286 0.04464286 0.0625    ]
   [0.08035714 0.04464286 0.         0.00892857]
   [0.01785714 0.0625     0.00892857 0.01785714]]

  [[0.14285714 0.12244898 0.06122449 0.02040816]
   [0.12244898 0.20408163 0.04081633 0.03061224]
   [0.06122449 0.04081633 0.         0.03061224]
   [0.02040816 0.03061224 0.03061224 0.04081633]]]]


In [13]:
tp_val_glcm = val_glcm[:,:,:,::-1].transpose(0,3,2,1)
print("GLCM matrix:\n", tp_val_glcm)

GLCM matrix:
 [[[[16. 14.  5.  4.]
   [14. 22.  3.  6.]
   [ 5.  3.  4.  2.]
   [ 4.  6.  2.  2.]]

  [[ 6. 16.  6.  5.]
   [16. 12.  7.  4.]
   [ 6.  7.  0.  1.]
   [ 5.  4.  1.  2.]]

  [[14. 18.  9.  2.]
   [18. 12.  5.  7.]
   [ 9.  5.  0.  1.]
   [ 2.  7.  1.  2.]]

  [[14. 12.  6.  2.]
   [12. 20.  4.  3.]
   [ 6.  4.  0.  3.]
   [ 2.  3.  3.  4.]]]]


## Set weightingNorm to 'no_weighting'

In [14]:
extractor.settings['weightingNorm'] = 'no_weighting'
# extractor.settings['weightingNorm'] = 'None'  # default

In [15]:
glcm_calc_N = RadiomicsGLCM(imageName, maskName, **extractor.settings)

In [16]:
# Check the GLCM coefficients
print("Ng (Num_grayLevels):", glcm_calc_N.coefficients['Ng'])
print("GrayLevels:", glcm_calc_N.coefficients['grayLevels'])

Ng (Num_grayLevels): 4
GrayLevels: [1 2 3 4]


In [17]:
glcm_calc_N.execute()  # calc GLCM

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


{'Autocorrelation': array(4.10952381),
 'ClusterProminence': array(12.53755185),
 'ClusterShade': array(1.97483598),
 'ClusterTendency': array(2.0131746),
 'Contrast': array(1.84285714),
 'Correlation': array(0.0441691),
 'DifferenceAverage': array(1.03333333),
 'DifferenceEntropy': array(1.77947293),
 'DifferenceVariance': array(0.77507937),
 'Id': array(0.60277778),
 'Idm': array(0.56428571),
 'Idmn': array(0.90981513),
 'Idn': array(0.81854875),
 'Imc1': array(-0.01100464),
 'Imc2': array(0.19778314),
 'InverseVariance': array(0.47116402),
 'JointAverage': array(2.01666667),
 'JointEnergy': array(0.09910431),
 'JointEntropy': array(3.6061393),
 'MCC': array(0.12543529),
 'MaximumProbability': array(0.15714286),
 'SumAverage': array(4.03333333),
 'SumEntropy': array(2.44303408),
 'SumSquares': array(0.96400794)}

In [18]:
original_P_glcm = glcm_calc.orginal_P_glcm
original_angles = glcm_calc.original_angles
print("Original GLCM shape:", original_P_glcm.shape)
print("Original GLCM matrix:\n", original_P_glcm)
print("Original angles:\n", original_angles)

Original GLCM shape: (1, 4, 4, 4)
Original GLCM matrix:
 [[[[ 7.  7.  3.  8.]
   [ 5.  6.  8.  7.]
   [ 3.  6.  3.  3.]
   [ 1.  2.  2.  1.]]

  [[ 7. 12.  8.  7.]
   [10.  6.  6. 11.]
   [ 3.  2.  4.  1.]
   [ 2.  3.  3.  4.]]

  [[ 3.  3.  3.  2.]
   [ 1.  3.  3.  2.]
   [ 0.  0.  0.  2.]
   [ 2.  1.  1.  1.]]

  [[ 1.  0.  3.  3.]
   [ 1.  4.  1.  2.]
   [ 1.  0.  0.  1.]
   [ 2.  1.  1.  1.]]]]
Original angles:
 [[ 0  1  1]
 [ 0  1  0]
 [ 0  1 -1]
 [ 0  0  1]]


In [19]:
p_glcm_N = glcm_calc_N.P_glcm  # get GLCM matrix [i,j,angle] with normalization 
print("GLCM shape:", p_glcm_N.shape)
print("GLCM matrix:\n", p_glcm_N)

GLCM shape: (1, 4, 4, 1)
GLCM matrix:
 [[[[0.11904762]
   [0.14285714]
   [0.06190476]
   [0.03095238]]

  [[0.14285714]
   [0.15714286]
   [0.0452381 ]
   [0.04761905]]

  [[0.06190476]
   [0.0452381 ]
   [0.00952381]
   [0.01666667]]

  [[0.03095238]
   [0.04761905]
   [0.01666667]
   [0.02380952]]]]


In [20]:
print(glcm_calc_N.sumP_glcm)
print(glcm_calc_N.raw_sumP_glcm)

[[420.]]
[[420.]]


#### Note!!
Cannot back to pre-normarized GLCM with weightingNorm = 'no_weighting' with same methods.

In [21]:
val_glcm_N = p_glcm_N * glcm_calc.sumP_glcm

In [24]:
sum_val_glcm_N = np.sum(val_glcm_N,axis=(0, 1, 2))
print("sum_val_glcm_N:\n", sum_val_glcm_N)
print("val_glcm_N:\n", val_glcm_N)

sum_val_glcm_N:
 [ 98. 112.  98. 112.]
val_glcm_N:
 [[[[11.66666667 13.33333333 11.66666667 13.33333333]
   [14.         16.         14.         16.        ]
   [ 6.06666667  6.93333333  6.06666667  6.93333333]
   [ 3.03333333  3.46666667  3.03333333  3.46666667]]

  [[14.         16.         14.         16.        ]
   [15.4        17.6        15.4        17.6       ]
   [ 4.43333333  5.06666667  4.43333333  5.06666667]
   [ 4.66666667  5.33333333  4.66666667  5.33333333]]

  [[ 6.06666667  6.93333333  6.06666667  6.93333333]
   [ 4.43333333  5.06666667  4.43333333  5.06666667]
   [ 0.93333333  1.06666667  0.93333333  1.06666667]
   [ 1.63333333  1.86666667  1.63333333  1.86666667]]

  [[ 3.03333333  3.46666667  3.03333333  3.46666667]
   [ 4.66666667  5.33333333  4.66666667  5.33333333]
   [ 1.63333333  1.86666667  1.63333333  1.86666667]
   [ 2.33333333  2.66666667  2.33333333  2.66666667]]]]


In [22]:
tp_p_glcm_N = p_glcm_N[:,:,:,::-1].transpose(0,3,2,1)
print("GLCM matrix:\n", tp_p_glcm_N)

GLCM matrix:
 [[[[0.11904762 0.14285714 0.06190476 0.03095238]
   [0.14285714 0.15714286 0.0452381  0.04761905]
   [0.06190476 0.0452381  0.00952381 0.01666667]
   [0.03095238 0.04761905 0.01666667 0.02380952]]]]


In [23]:
tp_val_glcm_N = val_glcm_N[:,:,:,::-1].transpose(0,3,2,1)
print("GLCM matrix:\n", tp_val_glcm_N)

GLCM matrix:
 [[[[13.33333333 16.          6.93333333  3.46666667]
   [16.         17.6         5.06666667  5.33333333]
   [ 6.93333333  5.06666667  1.06666667  1.86666667]
   [ 3.46666667  5.33333333  1.86666667  2.66666667]]

  [[11.66666667 14.          6.06666667  3.03333333]
   [14.         15.4         4.43333333  4.66666667]
   [ 6.06666667  4.43333333  0.93333333  1.63333333]
   [ 3.03333333  4.66666667  1.63333333  2.33333333]]

  [[13.33333333 16.          6.93333333  3.46666667]
   [16.         17.6         5.06666667  5.33333333]
   [ 6.93333333  5.06666667  1.06666667  1.86666667]
   [ 3.46666667  5.33333333  1.86666667  2.66666667]]

  [[11.66666667 14.          6.06666667  3.03333333]
   [14.         15.4         4.43333333  4.66666667]
   [ 6.06666667  4.43333333  0.93333333  1.63333333]
   [ 3.03333333  4.66666667  1.63333333  2.33333333]]]]
